## `01 — Exploration Gmail & Baseline SVM`

Ce notebook couvre :
1. Connexion à l'API Gmail et récupération des mails
2. Prétraitement du texte
3. Visualisation (t-SNE / UMAP)
4. Baseline classifier (TF-IDF + SVM)
5. Évaluation et matrice de confusion

### `0. Installation des dépendances`

In [ ]:
# À exécuter une seule fois
#%pip -q install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
#%pip -q install scikit-learn pandas numpy matplotlib seaborn umap-learn
%pip -q install imapclient mail-parser tqdm

Note: you may need to restart the kernel to use updated packages.


### `1. Connexion Scaleway API`

In [13]:
from imapclient import IMAPClient
import mailparser
import pandas as pd
from tqdm import tqdm

IMAP_HOST   = "imap.online.net"
IMAP_USER   = "gerard@benitah.net"
IMAP_PASS   = "O~Eqco_#U>x_J@0^/5"
IMAP_PORT   = 993

FOLDER_HAM  = "INBOX"
FOLDER_SPAM = "INBOX/SPAM"   # ou "Junk", "Pourriel", selon ton serveur

def fetch_folder(folder, label, limit=None):
    rows = []
    with IMAPClient(IMAP_HOST, ssl=True) as client:
        client.login(IMAP_USER, IMAP_PASS)
        for flags, delim, name in client.list_folders():
            print(repr(name))


        print("Folders:")
        for flags, delim, name in client.list_folders():
            print(name)

        # sélectionner le bon dossier
        client.select_folder(folder)  # ex: "INBOX" ou "Junk"

        uids = client.search("ALL")   # ou ["ALL"], les deux passent
        if limit:
            uids = uids[:limit]

        for uid in uids:
            msg_bytes = client.fetch([uid], ["RFC822"])[uid][b"RFC822"]
            mail = mailparser.parse_from_bytes(msg_bytes)
            subject = mail.subject or ""
            body = mail.text_plain[0] if mail.text_plain else (mail.body or "")
            text = (subject + " " + body).strip()
            if text:
                rows.append({"text": text, "label": label})
    return rows

rows_ham  = fetch_folder(FOLDER_HAM,  0, limit=2000)  # 0 = ham
rows_spam = fetch_folder(FOLDER_SPAM, 1, limit=2000)  # 1 = spam

df = pd.DataFrame(rows_ham + rows_spam).sample(frac=1, random_state=42)
df.to_csv("emails_spam_ham.csv", index=False)
print(df["label"].value_counts())

'vélo'
'Sent Messages'
'Drafts'
'INBOX/Drafts'
'INBOX/Sent'
'INBOX/SPAM'
'INBOX/Trash'
'Deleted Messages'
'job'
'SNCF'
'garder'
'Notes'
'achat'
'Junk'
'personnel'
'INBOX'
Folders:
vélo
Sent Messages
Drafts
INBOX/Drafts
INBOX/Sent
INBOX/SPAM
INBOX/Trash
Deleted Messages
job
SNCF
garder
Notes
achat
Junk
personnel
INBOX
'vélo'
'Sent Messages'
'Drafts'
'INBOX/Drafts'
'INBOX/Sent'
'INBOX/SPAM'
'INBOX/Trash'
'Deleted Messages'
'job'
'SNCF'
'garder'
'Notes'
'achat'
'Junk'
'personnel'
'INBOX'
Folders:
vélo
Sent Messages
Drafts
INBOX/Drafts
INBOX/Sent
INBOX/SPAM
INBOX/Trash
Deleted Messages
job
SNCF
garder
Notes
achat
Junk
personnel
INBOX


Email content 'calendar' not handled


label
1    2000
0      82
Name: count, dtype: int64


### `1. Connexion Gmail API`

In [ ]:
import os
import base64
import pandas as pd
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly','https://www.googleapis.com/auth/gmail.labels']

def get_gmail_service():
    """Authentification OAuth2 et création du service Gmail."""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Télécharger credentials.json depuis Google Cloud Console
            flow = InstalledAppFlow.from_client_secrets_file('credential.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

service = get_gmail_service()
print('✓ Connecté à Gmail')

✓ Connecté à Gmail


### `2. Création des labels d'entraînement`

Deux labels Gmail seront créés :
- `SPAM_TRAINING` → mails que tu identifies comme spam
- `HAM_TRAINING` → mails légitimes

Tu peux les alimenter manuellement depuis Gmail avant de lancer la suite.

In [14]:
def create_label_if_not_exists(service, name):
    """Crée un label Gmail s'il n'existe pas déjà."""
    existing = service.users().labels().list(userId='me').execute().get('labels', [])
    for label in existing:
        if label['name'] == name:
            print(f'Label "{name}" déjà existant (id={label["id"]})')
            return label['id']
    new_label = service.users().labels().create(
        userId='me',
        body={'name': name, 'labelListVisibility': 'labelShow', 'messageListVisibility': 'show'}
    ).execute()
    print(f'✓ Label "{name}" créé (id={new_label["id"]})')
    return new_label['id']

spam_label_id = create_label_if_not_exists(service, 'SPAM_TRAINING')
ham_label_id  = create_label_if_not_exists(service, 'HAM_TRAINING')

✓ Label "SPAM_TRAINING" créé (id=Label_14)
✓ Label "HAM_TRAINING" créé (id=Label_15)


### `3. Récupération des mails labellisés`

In [15]:
def fetch_emails_by_label(service, label_id, max_results=500):
    """Récupère les emails d'un label donné et retourne une liste de dicts."""
    messages_ref = service.users().messages().list(
        userId='me', labelIds=[label_id], maxResults=max_results
    ).execute().get('messages', [])

    emails = []
    for msg_ref in messages_ref:
        msg = service.users().messages().get(
            userId='me', id=msg_ref['id'], format='full'
        ).execute()

        # Extraction des headers
        headers = {h['name']: h['value'] for h in msg['payload'].get('headers', [])}
        subject = headers.get('Subject', '')
        sender  = headers.get('From', '')

        # Extraction du corps (text/plain en priorité)
        body = ''
        parts = msg['payload'].get('parts', [msg['payload']])
        for part in parts:
            if part.get('mimeType') == 'text/plain':
                data = part.get('body', {}).get('data', '')
                if data:
                    body = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
                    break

        emails.append({'subject': subject, 'sender': sender, 'body': body})

    return emails

spam_emails = fetch_emails_by_label(service, spam_label_id)
ham_emails  = fetch_emails_by_label(service, ham_label_id)

print(f'Spam récupérés : {len(spam_emails)}')
print(f'Ham récupérés  : {len(ham_emails)}')

Spam récupérés : 0
Ham récupérés  : 0


### `4. Construction du DataFrame`

In [24]:
def build_text(df_local):
    cols = df_local.columns.str.lower()
    # on cherche un candidat raisonnable pour le texte
    if 'body' in cols:
        col = df_local.columns[cols.get_loc('body')]
    elif 'subject' in cols:
        col = df_local.columns[cols.get_loc('subject')]
    elif 'snippet' in cols:
        col = df_local.columns[cols.get_loc('snippet')]
    else:
        raise ValueError(f"Aucune colonne texte trouvée dans {df_local.columns.tolist()}")
    df_local['text'] = df_local[col].fillna('').astype(str).str.strip()

In [27]:
import pandas as pd

df_spam = pd.DataFrame(spam_emails)
df_spam['label'] = 1  # 1 = spam

df_ham = pd.DataFrame(ham_emails)
df_ham['label'] = 0  # 0 = ham


print(df_spam.head())
print(df_spam.columns.tolist())

print(df_ham.head())
print(df_ham.columns.tolist())

# Construire la colonne text dans chaque DF
build_text(df_spam)
build_text(df_ham)

print("df_spam columns:", df_spam.columns.tolist())
print("df_ham columns:", df_ham.columns.tolist())


df = pd.concat([df_spam, df_ham], ignore_index=True).sample(frac=1, random_state=42)

# Texte complet = sujet + corps
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
df['text'] = df['text'].str.strip()

print(df[['label', 'subject']].groupby('label').count())
df.head(3)

Empty DataFrame
Columns: [label]
Index: []
['label']
Empty DataFrame
Columns: [label]
Index: []
['label']


ValueError: Aucune colonne texte trouvée dans ['label']

### `5. Prétraitement du texte`

Nettoyage minimal : lowercase, suppression URLs, ponctuation.

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)   # URLs
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)          # Adresses email
    text = re.sub(r'[^\w\s]', ' ', text)                 # Ponctuation
    text = re.sub(r'\s+', ' ', text).strip()             # Espaces multiples
    return text

df['text_clean'] = df['text'].apply(clean_text)
df['text_clean'].head(3)

### `6. Vectorisation TF-IDF + Baseline SVM`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

X = df['text_clean'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF avec n-grammes (unigrammes + bigrammes)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# SVM linéaire (rapide et efficace sur texte)
clf = LinearSVC(C=1.0, max_iter=2000)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

### `7. Matrice de confusion`

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.title('Matrice de confusion — SVM baseline')
plt.tight_layout()
plt.savefig('confusion_matrix_svm.png', dpi=150)
plt.show()

### `8. Visualisation t-SNE des embeddings TF-IDF`

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD

# Réduction dimensionnelle préalable (TF-IDF est sparse, t-SNE ne gère pas >200 dims)
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(vectorizer.transform(df['text_clean'].values))

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d = tsne.fit_transform(X_reduced)

plt.figure(figsize=(8, 6))
colors = ['steelblue' if l == 0 else 'tomato' for l in df['label'].values]
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, alpha=0.6, s=20)
plt.legend(handles=[
    plt.scatter([], [], c='steelblue', label='Ham'),
    plt.scatter([], [], c='tomato', label='Spam')
])
plt.title('t-SNE — TF-IDF embeddings')
plt.tight_layout()
plt.savefig('tsne_tfidf.png', dpi=150)
plt.show()

### `9. Sauvegarde du dataset et du modèle baseline`

Pour les réutiliser dans le notebook 02.

In [ ]:
import joblib

df.to_csv('dataset.csv', index=False)
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(clf, 'svm_baseline.pkl')
print('✓ Dataset et modèle sauvegardés')